In [ ]:
from environment import Environment
from collections import defaultdict
import pandas as pd
import os
import matplotlib.pyplot as plt
import math
import random
import numpy as np

### Yellow Warnings

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N = 1000000
BATCH_SIZE = 10000

for warning in ['yellow']:#, 'amber', 'red']:
    print (warning)
    csv_path = f"warning/samples/{warning}_samples.csv"
    environment = Environment(use_historic=warning)
    header_written = os.path.exists(csv_path)

    batch = defaultdict(list)

    for i in range(N):
        print(i)
        # --------------- Sampling
        environment.sample_features()
        environment.update_derived()

        # --------------- Saving samples
        for key, value in environment.samples.features.items():
            if key not in ("season", "geometry"):
                batch[key].append(value)

        for key, value in environment.derived.items():
            batch[key].append(value)

        batch["impact"].append(environment.impact)

        # Flush to disk every BATCH_SIZE iterations
        if (i + 1) % BATCH_SIZE == 0:
            print(f"Batch {(i + 1) // BATCH_SIZE}")
            df = pd.DataFrame(batch)
            df.to_csv(csv_path, mode="a", index=False, header=not header_written)
            header_written = True
            batch = defaultdict(list)

    # Write any remaining rows
    if batch:
        df = pd.DataFrame(batch)
        df.to_csv(csv_path, mode="a", index=False, header=not header_written)

In [ ]:
warning = "yellow"

df = pd.read_csv(f"warning/samples/{warning}_samples.csv")

n = len(df.columns)

ncols = math.ceil(math.sqrt(n))
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()

for ax, col in zip(axes, df.columns):
    ax.hist(df[col].dropna(), bins=30)
    ax.set_title(col, fontsize=8)

# hide unused axes
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
df['impact'].hist()
plt.title(f'{warning} warnings Impact Distribution')
plt.xlabel('Impact')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
print(df['impact'].describe())